# Benchmark — Qwen 0.5B: Base vs DPO vs ORPO

Valuta tre modelli sugli stessi benchmark per confrontare l'effetto del finetuning.

**Benchmark inclusi:**
- **MMLU** — conoscenza generale (baseline di controllo: non deve degradare)
- **TruthfulQA** — calibrazione e onestà delle risposte
- **IFEval** — instruction following (il più rilevante per DPO/ORPO su istruzioni generali)

**Strumento:** `lm-evaluation-harness` di EleutherAI — standard de facto, gira tutto in locale senza API.

> ⚠️ Assicurati di usare **T4 GPU** in `Settings > Accelerator`.

## Cella 1 — Installazione

In [1]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "lm-eval[ifeval]",   # include dipendenze per IFEval
    "unsloth[kaggle-new]",
    "--quiet", "--upgrade"
])

import lm_eval, unsloth
print(f"✅ lm-eval  {lm_eval.__version__}")
print(f"✅ unsloth  {unsloth.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.3/983.3 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ lm-eval  0.4.12
✅ unsloth  2026.6.2


## Cella 2 — Setup ambiente

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"]    = "0"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_DIR  = "/kaggle/working"
CACHE_DIR = os.path.join(BASE_DIR, "hf_cache")
os.environ["HF_HOME"]           = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "transformers")
os.environ["HF_DATASETS_CACHE"]  = os.path.join(CACHE_DIR, "datasets")
os.makedirs(CACHE_DIR, exist_ok=True)

import torch
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU  : Tesla T4
VRAM : 14.6 GB


## Cella 3 — Configurazione

**Modifica `MODEL_CHOICE` per scegliere cosa valutare.**
Esegui questa cella + la Cella 4 + la Cella 5 per ogni modello che vuoi confrontare.

In [9]:
# ─────────────────────────────────────────────────────────────
#  SCEGLI IL MODELLO DA VALUTARE
#  Opzioni: "base" | "dpo" | "orpo"
# ─────────────────────────────────────────────────────────────
MODEL_CHOICE = "orpo"   # ← cambia qui

# ── Percorsi adapter su Kaggle ────────────────────────────────
ADAPTER_PATHS = {
    "dpo":  "/kaggle/input/datasets/lorenzosalis/qwen-0-5b-dpo/Qwen DPO",
    "orpo": "/kaggle/input/datasets/lorenzosalis/qwen-0-5b-orpo/Qwen ORPO",
}

# ── Modello base ──────────────────────────────────────────────
BASE_MODEL_ID = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

# ── Benchmark ─────────────────────────────────────────────────
TASKS = [
    "arc_challenge",    # reasoning — baseline di controllo
    "hellaswag",        # commonsense — baseline di controllo
    "winogrande",       # commonsense — baseline di controllo
    "truthfulqa_mc2",   # calibrazione — leggero miglioramento atteso
    "mmlu",             # conoscenza — non deve degradare
    #"ifeval",           # instruction following — miglioramento principale atteso
]

METRIC_MAP = {
    "arc_challenge":  "acc_norm,none",
    "hellaswag":      "acc_norm,none",
    "winogrande":     "acc,none",
    "truthfulqa_mc2": "acc,none",
    "mmlu":           "acc,none",
    "ifeval":         "prompt_level_strict_acc,none",
}

# Baseline ufficiali Qwen2.5-0.5B-Instruct (0-shot, test set completo)
OFFICIAL_BASELINE = {
    "arc_challenge":  0.339,
    "hellaswag":      0.409,
    "winogrande":     0.553,
    "truthfulqa_mc2": 0.419,
    "mmlu":           0.474,
    "ifeval":         0.229,
}

# ── Parametri valutazione ─────────────────────────────────────
BATCH_SIZE     = 4    # riduci a 2 se OOM
MAX_NEW_TOKENS = 256
NUM_FEWSHOT    = 0    # 0-shot su tutti: coerente con le baseline ufficiali sopra
LIMIT          = 500 

# ── Output ────────────────────────────────────────────────────
RESULTS_DIR = os.path.join("/kaggle/working", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"🎯 Modello selezionato : {MODEL_CHOICE.upper()}")
print(f"📋 Benchmark           : {TASKS}")
print(f"📁 Output risultati    : {RESULTS_DIR}")

🎯 Modello selezionato : ORPO
📋 Benchmark           : ['arc_challenge', 'hellaswag', 'winogrande', 'truthfulqa_mc2', 'mmlu']
📁 Output risultati    : /kaggle/working/results


## Cella 4 — Caricamento modello

In [10]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

torch.cuda.empty_cache()

print(f"⏳ Caricamento modello '{MODEL_CHOICE}'...")

# Carica sempre il modello base
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
    cache_dir      = CACHE_DIR,
)

# Se non è "base", carica l'adapter LoRA sopra
if MODEL_CHOICE in ("dpo", "orpo"):
    adapter_path = ADAPTER_PATHS[MODEL_CHOICE]
    if not os.path.exists(os.path.join(adapter_path, "adapter_config.json")):
        raise FileNotFoundError(
            f"adapter_config.json non trovato in: {adapter_path}\n"
            f"Verifica che il dataset Kaggle sia montato correttamente."
        )
    model = PeftModel.from_pretrained(model, adapter_path)
    model = model.merge_and_unload()   # merge adapter nei pesi base per inferenza più veloce
    print(f"   ✅ Adapter {MODEL_CHOICE.upper()} caricato e merged")

# Metti in modalità inferenza
FastLanguageModel.for_inference(model)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"✅ Modello pronto — VRAM usata: {vram_used:.2f} GB")

⏳ Caricamento modello 'orpo'...
==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


   ✅ Adapter ORPO caricato e merged
✅ Modello pronto — VRAM usata: 1.34 GB


## Cella 5 — Esecuzione benchmark

Esegue tutti i task in `TASKS` e salva i risultati in un file JSON nominato col modello scelto.

**Tempi stimati su T4:**
- MMLU (0-shot): ~20-25 min
- TruthfulQA: ~5 min
- IFEval: ~10-15 min

In [11]:
import json
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

torch.cuda.empty_cache()

# Wrappa il modello nel formato atteso da lm-eval
lm_model = HFLM(
    pretrained    = model,
    tokenizer     = tokenizer,
    batch_size    = BATCH_SIZE,
    max_length    = 2048,
    dtype         = "float16",
)

print(f"🚀 Avvio benchmark per modello: {MODEL_CHOICE.upper()}")
print(f"   Task: {TASKS}")
print(f"   Few-shot: {NUM_FEWSHOT}-shot\n")

results = evaluator.simple_evaluate(
    model          = lm_model,
    tasks          = TASKS,
    num_fewshot    = NUM_FEWSHOT,
    batch_size     = BATCH_SIZE,
    device         = "cuda",
    log_samples    = False,
    limit          = LIMIT
    #gen_kwargs     = "max_new_tokens=64",
)

# ── Salva risultati completi su file ──────────────────────────
output_file = os.path.join(RESULTS_DIR, f"results_{MODEL_CHOICE}.json")
with open(output_file, "w") as f:
    json.dump(results["results"], f, indent=2)
print(f"\n💾 Risultati salvati in: {output_file}")

# ── Stampa riepilogo leggibile ────────────────────────────────
print(f"\n{'='*55}")
print(f"  RISULTATI — {MODEL_CHOICE.upper()}")
print(f"{'='*55}")

# Metriche principali per task
METRIC_MAP = {
    "mmlu":           "acc,none",
    "truthfulqa_mc2": "acc,none",
    "ifeval":         "prompt_level_strict_acc,none",
}

for task, metric_key in METRIC_MAP.items():
    if task in results["results"]:
        val = results["results"][task].get(metric_key, "N/A")
        if isinstance(val, float):
            print(f"  {task:<20} {metric_key:<35} {val*100:.2f}%")
        else:
            print(f"  {task:<20} {metric_key:<35} {val}")

print(f"{'='*55}")

[lm_eval.models.huggingface|WARNING]`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
[lm_eval.models.huggingface|WARNING]Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


🚀 Avvio benchmark per modello: ORPO
   Task: ['arc_challenge', 'hellaswag', 'winogrande', 'truthfulqa_mc2', 'mmlu']
   Few-shot: 0-shot



[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of arc_challenge from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of hellaswag from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of winogrande from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_abstract_algebra from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_anatomy from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_astronomy from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_biology from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_chemistry from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_computer_science from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_mathematics from None to 0
[lm_eval.evaluator|WARNING]Overwriting default


💾 Risultati salvati in: /kaggle/working/results/results_orpo.json

  RISULTATI — ORPO
  mmlu                 acc,none                            45.72%
  truthfulqa_mc2       acc,none                            42.12%


## Cella 6 — Confronto finale tra modelli

Esegui questa cella **dopo** aver valutato tutti e tre i modelli (base, dpo, orpo).
Carica i tre file JSON salvati e produce una tabella comparativa.

In [15]:
import json, os
import pandas as pd

MODELS = ["base", "dpo", "orpo"]

METRIC_MAP_DISPLAY = {
    "arc_challenge":  "acc_norm,none",
    "hellaswag":      "acc_norm,none",
    "winogrande":     "acc,none",
    "truthfulqa_mc2": "acc,none",
    "mmlu":           "acc,none",
}

OFFICIAL_BASELINE = {
    "arc_challenge":  0.339,
    "hellaswag":      0.409,
    "winogrande":     0.553,
    "truthfulqa_mc2": 0.419,
    "mmlu":           0.474,
}

rows = []
for model_name in MODELS:
    path = os.path.join(RESULTS_DIR, f"results_{model_name}.json")
    if not os.path.exists(path):
        print(f"⚠️  File non trovato per '{model_name}' — skippato")
        continue
    with open(path) as f:
        res = json.load(f)

    row = {"modello": model_name.upper()}
    for task, metric_key in METRIC_MAP_DISPLAY.items():
        if task not in res:
            row[task] = "—"
            continue
        # prova prima il metric_key esatto, poi varianti comuni
        val = (res[task].get(metric_key)
               or res[task].get(metric_key.split(",")[0])
               or res[task].get("acc,none")
               or res[task].get("acc_norm,none"))
        row[task] = round(val * 100, 2) if isinstance(val, float) else "—"
    rows.append(row)

if not rows:
    print("Nessun risultato trovato.")
else:
    df = pd.DataFrame(rows).set_index("modello")

    # ── Delta interni ─────────────────────────────────────────
    if "BASE" in df.index:
        for m in ["DPO", "ORPO"]:
            if m in df.index:
                delta = {}
                for col in df.columns:
                    try:
                        delta[col] = round(float(df.loc[m, col]) - float(df.loc["BASE", col]), 2)
                    except (ValueError, TypeError):
                        delta[col] = "—"
                df.loc[f"Δ {m} vs BASE"] = delta

    print("\n" + "="*70)
    print("  CONFRONTO TRA MODELLI (accuracy %)")
    print("="*70)
    print(df.to_string())
    print("Δ positivo = miglioramento rispetto al modello base interno\n")

    # ── Confronto con baseline ufficiale ─────────────────────
    print("="*70)
    print("  CONFRONTO CON BASELINE UFFICIALE Qwen2.5-0.5B-Instruct")
    print("="*70)
    print(f"{'Task':<20} {'Ufficiale':>10} {'BASE':>8} {'DPO':>8} {'ORPO':>8} {'Δ DPO':>8} {'Δ ORPO':>8}")
    print("-"*70)

    for task, official in OFFICIAL_BASELINE.items():
        line = f"{task:<20} {official*100:>9.1f}%"
        scores = {}
        for m in MODELS:
            key = m.upper()
            val = df.loc[key, task] if key in df.index else None
            try:
                scores[m] = float(val)
                line += f" {val:>7.2f}%"
            except (ValueError, TypeError):
                scores[m] = None
                line += f" {'—':>8}"
        for m in ["dpo", "orpo"]:
            if scores.get(m) is not None:
                delta = scores[m] - official * 100
                arrow = "▲" if delta > 0 else "▼"
                line += f"  {arrow}{abs(delta):>4.1f}%"
            else:
                line += f" {'—':>8}"
        print(line)

    print("="*70)

    csv_path = os.path.join(RESULTS_DIR, "confronto_finale.csv")
    df.to_csv(csv_path)
    print(f"\n💾 Tabella salvata in: {csv_path}")


  CONFRONTO TRA MODELLI (accuracy %)
                arc_challenge  hellaswag  winogrande  truthfulqa_mc2   mmlu
modello                                                                    
BASE                     32.4       49.2        59.2           42.11  45.81
DPO                      32.4       49.2        59.2           42.07  45.77
ORPO                     32.4       49.4        58.8           42.12  45.72
Δ DPO vs BASE             0.0        0.0         0.0           -0.04  -0.04
Δ ORPO vs BASE            0.0        0.2        -0.4            0.01  -0.09
Δ positivo = miglioramento rispetto al modello base interno

  CONFRONTO CON BASELINE UFFICIALE Qwen2.5-0.5B-Instruct
Task                  Ufficiale     BASE      DPO     ORPO    Δ DPO   Δ ORPO
----------------------------------------------------------------------
arc_challenge             33.9%   32.40%   32.40%   32.40%  ▼ 1.5%  ▼ 1.5%
hellaswag                 40.9%   49.20%   49.20%   49.40%  ▲ 8.3%  ▲ 8.5%
winogrande    